# Trade Analysis & Labeling Workflow

This notebook helps you:
1. Load trades from training logs
2. Analyze trade patterns and VP context
3. Filter and visualize trades
4. Prepare trades for manual labeling
5. Analyze labeled trades to guide reward shaping

## 1. Setup & Imports

In [ ]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import json
sys.path.append('src')

from features.visible_range_vp import VisibleRangeVP

print("✓ Imports complete")

## 2. Load Trades from Log Files

In [ ]:
# List all available trade log files
log_dir = Path('logs/trades')
log_dir.mkdir(parents=True, exist_ok=True)

trade_files = sorted(log_dir.glob('trades_*.pkl'))

print(f"Found {len(trade_files)} trade log files:")
for i, f in enumerate(trade_files, 1):
    # Get file size
    size_kb = f.stat().st_size / 1024
    print(f"  {i}. {f.name} ({size_kb:.1f} KB)")

In [ ]:
# Load the latest (or specific) trade file
if trade_files:
    # Use latest file
    selected_file = trade_files[-1]
    
    # Or select specific file by index (uncomment and adjust):
    # selected_file = trade_files[0]
    
    with open(selected_file, 'rb') as f:
        trades = pickle.load(f)
    
    print(f"✓ Loaded {len(trades)} trades from {selected_file.name}")
else:
    print("⚠️ No trade files found. Run training with enable_trade_logging=True first.")
    trades = []

## 2b. Load Market Data (OHLCV)

In [ ]:
from utils.indicator_utils import add_indicators

DATA_SYMBOL = 'BTCUSDT'
DATA_TIMEFRAME = '5m'
DATA_PATH = f'data/binance-{DATA_SYMBOL}-{DATA_TIMEFRAME}.pkl'

# Load the actual market data for visualization
data_file = Path(DATA_PATH)

if data_file.exists():
    df = pd.read_pickle(DATA_PATH)
    df['date'] = pd.to_datetime(df['date_close'])
    add_indicators(df)
    df_market = df.dropna().reset_index(drop=True)
    
    print(f"✓ Loaded {len(df_market)} candles from {data_file.name}")
    print(f"Columns: {df_market.columns.tolist()}")
    print(f"\nFirst few candles:")
    display(df_market.head())
else:
    print(f"⚠️ Market data file not found: {data_file}")
    df_market = pd.DataFrame()

## 3. Convert to DataFrame for Analysis

In [ ]:
if trades:
    df_trades = pd.DataFrame(trades)
    
    print(f"Trade columns: {df_trades.columns.tolist()}")
    print(f"\nDataFrame shape: {df_trades.shape}")
    print(f"\nFirst few trades:")
    display(df_trades.head())
else:
    df_trades = pd.DataFrame()

## 4. Basic Trade Statistics

In [ ]:
if not df_trades.empty:
    print("="*60)
    print("TRADE STATISTICS")
    print("="*60)
    
    # Action distribution
    print("\n📊 Action Distribution:")
    action_counts = df_trades['action'].value_counts()
    for action, count in action_counts.items():
        pct = (count / len(df_trades)) * 100
        print(f"  {action}: {count} ({pct:.1f}%)")
    
    # P&L statistics
    print("\n💰 P&L Statistics:")
    print(f"  Total P&L: ${df_trades['pnl'].sum():.2f}")
    print(f"  Avg P&L per trade: ${df_trades['pnl'].mean():.2f}")
    print(f"  Avg P&L %: {df_trades['pnl_pct'].mean():.2f}%")
    print(f"  Best trade: ${df_trades['pnl'].max():.2f} ({df_trades['pnl_pct'].max():.2f}%)")
    print(f"  Worst trade: ${df_trades['pnl'].min():.2f} ({df_trades['pnl_pct'].min():.2f}%)")
    
    # Win rate
    winners = (df_trades['pnl'] > 0).sum()
    losers = (df_trades['pnl'] < 0).sum()
    win_rate = (winners / len(df_trades)) * 100 if len(df_trades) > 0 else 0
    
    print("\n📈 Performance:")
    print(f"  Winners: {winners} ({(winners/len(df_trades)*100):.1f}%)")
    print(f"  Losers: {losers} ({(losers/len(df_trades)*100):.1f}%)")
    print(f"  Win Rate: {win_rate:.1f}%")
    
    # Profit factor
    gross_profit = df_trades[df_trades['pnl'] > 0]['pnl'].sum()
    gross_loss = abs(df_trades[df_trades['pnl'] < 0]['pnl'].sum())
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')
    print(f"  Profit Factor: {profit_factor:.2f}")
    
    # Duration
    print("\n⏱️ Duration:")
    print(f"  Avg duration: {df_trades['duration'].mean():.1f} candles")
    print(f"  Min duration: {df_trades['duration'].min()} candles")
    print(f"  Max duration: {df_trades['duration'].max()} candles")

## 5. Volume Profile Context Analysis

In [ ]:
if not df_trades.empty:
    print("="*60)
    print("VOLUME PROFILE CONTEXT ANALYSIS")
    print("="*60)
    
    # Binary feature activation rates
    print("\n🔢 Binary Features (% of trades):")
    print(f"  close_in_va: {df_trades['close_in_va'].mean()*100:.1f}%")
    print(f"  close_above_va: {df_trades['close_above_va'].mean()*100:.1f}%")
    print(f"  close_below_va: {df_trades['close_below_va'].mean()*100:.1f}%")
    print(f"  wick_touched_va: {df_trades['wick_touched_va'].mean()*100:.1f}%")
    print(f"  close_above_poc: {df_trades['close_above_poc'].mean()*100:.1f}%")
    print(f"  wick_crossed_poc: {df_trades['wick_crossed_poc'].mean()*100:.1f}%")
    
    # Distance statistics by action
    print("\n📏 Average VP Distances by Action:")
    for action in df_trades['action'].unique():
        action_trades = df_trades[df_trades['action'] == action]
        print(f"\n  {action}:")
        print(f"    dist_to_vah: {action_trades['dist_to_vah'].mean():.3f}")
        print(f"    dist_to_poc: {action_trades['dist_to_poc'].mean():.3f}")
        print(f"    dist_to_val: {action_trades['dist_to_val'].mean():.3f}")
    
    # Winners vs Losers VP context
    print("\n🎯 VP Context: Winners vs Losers:")
    winners_df = df_trades[df_trades['pnl'] > 0]
    losers_df = df_trades[df_trades['pnl'] < 0]
    
    if len(winners_df) > 0 and len(losers_df) > 0:
        print("\n  Winners:")
        print(f"    close_in_va: {winners_df['close_in_va'].mean()*100:.1f}%")
        print(f"    dist_to_poc (avg): {winners_df['dist_to_poc'].mean():.3f}")
        print(f"    dist_to_val (avg): {winners_df['dist_to_val'].mean():.3f}")
        
        print("\n  Losers:")
        print(f"    close_in_va: {losers_df['close_in_va'].mean()*100:.1f}%")
        print(f"    dist_to_poc (avg): {losers_df['dist_to_poc'].mean():.3f}")
        print(f"    dist_to_val (avg): {losers_df['dist_to_val'].mean():.3f}")

## 6. Visualize Trade Distributions

In [ ]:
if not df_trades.empty:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # P&L Distribution
    axes[0, 0].hist(df_trades['pnl'], bins=30, edgecolor='black', alpha=0.7)
    axes[0, 0].axvline(0, color='red', linestyle='--', linewidth=2)
    axes[0, 0].set_xlabel('P&L ($)')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].set_title('P&L Distribution')
    axes[0, 0].grid(True, alpha=0.3)
    
    # P&L % Distribution
    axes[0, 1].hist(df_trades['pnl_pct'], bins=30, edgecolor='black', alpha=0.7, color='orange')
    axes[0, 1].axvline(0, color='red', linestyle='--', linewidth=2)
    axes[0, 1].set_xlabel('P&L %')
    axes[0, 1].set_ylabel('Count')
    axes[0, 1].set_title('P&L % Distribution')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Duration Distribution
    axes[0, 2].hist(df_trades['duration'], bins=30, edgecolor='black', alpha=0.7, color='green')
    axes[0, 2].set_xlabel('Duration (candles)')
    axes[0, 2].set_ylabel('Count')
    axes[0, 2].set_title('Trade Duration')
    axes[0, 2].grid(True, alpha=0.3)
    
    # Distance to POC
    axes[1, 0].hist(df_trades['dist_to_poc'], bins=30, edgecolor='black', alpha=0.7, color='purple')
    axes[1, 0].axvline(0, color='yellow', linestyle='--', linewidth=2, label='POC')
    axes[1, 0].set_xlabel('Distance to POC')
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].set_title('Entry Distance to POC')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Distance to VAL
    axes[1, 1].hist(df_trades['dist_to_val'], bins=30, edgecolor='black', alpha=0.7, color='cyan')
    axes[1, 1].axvline(0, color='blue', linestyle='--', linewidth=2, label='VAL')
    axes[1, 1].set_xlabel('Distance to VAL')
    axes[1, 1].set_ylabel('Count')
    axes[1, 1].set_title('Entry Distance to VAL')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # Distance to VAH
    axes[1, 2].hist(df_trades['dist_to_vah'], bins=30, edgecolor='black', alpha=0.7, color='red')
    axes[1, 2].axvline(0, color='blue', linestyle='--', linewidth=2, label='VAH')
    axes[1, 2].set_xlabel('Distance to VAH')
    axes[1, 2].set_ylabel('Count')
    axes[1, 2].set_title('Entry Distance to VAH')
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 7. Filter Trades for Labeling

Select interesting trades to label manually:

In [ ]:
if not df_trades.empty:
    # Filter criteria - adjust as needed
    
    # Option 1: Select profitable trades for "good entry" examples
    good_candidates = df_trades[
        (df_trades['pnl_pct'] > 1.0) &  # >1% profit
        (df_trades['duration'] > 5)     # Held for at least 5 candles
    ].copy()
    
    # Option 2: Select losing trades for "bad entry" examples
    bad_candidates = df_trades[
        (df_trades['pnl_pct'] < -0.5) &  # >0.5% loss
        (df_trades['duration'] > 5)
    ].copy()
    
    # Option 3: Select trades near VP levels
    vp_candidates = df_trades[
        (abs(df_trades['dist_to_poc']) < 0.1) |  # Near POC
        (abs(df_trades['dist_to_val']) < 0.1) |  # Near VAL
        (abs(df_trades['dist_to_vah']) < 0.1)    # Near VAH
    ].copy()
    
    print(f"✓ Good entry candidates: {len(good_candidates)}")
    print(f"✓ Bad entry candidates: {len(bad_candidates)}")
    print(f"✓ Trades near VP levels: {len(vp_candidates)}")
    
    # Combine and deduplicate
    candidates = pd.concat([good_candidates, bad_candidates, vp_candidates]).drop_duplicates(subset=['timestamp'])
    print(f"\n✓ Total unique candidates: {len(candidates)}")

## 8. Sample Trades for Quick Review

In [ ]:
if not df_trades.empty and len(candidates) > 0:
    # Show sample of candidates
    sample_size = min(10, len(candidates))
    sample = candidates.sample(n=sample_size)
    
    print(f"Sample of {sample_size} candidate trades for labeling:\n")
    
    for idx, trade in sample.iterrows():
        print(f"Trade #{idx}:")
        print(f"  Action: {trade['action']}")
        print(f"  Entry: ${trade['entry_price']:.2f} @ step {trade['timestamp']}")
        print(f"  Exit: ${trade['exit_price']:.2f}")
        print(f"  P&L: ${trade['pnl']:.2f} ({trade['pnl_pct']:.2f}%)")
        print(f"  Duration: {trade['duration']} candles")
        print(f"  VP Context:")
        print(f"    close_in_va: {trade['close_in_va']}")
        print(f"    dist_to_poc: {trade['dist_to_poc']:.3f}")
        print(f"    dist_to_val: {trade['dist_to_val']:.3f}")
        print(f"    dist_to_vah: {trade['dist_to_vah']:.3f}")
        print()

## 9. Export Candidates for Labeling

In [ ]:
if not df_trades.empty and len(candidates) > 0:
    # Convert back to list of dicts for pickle
    candidates_list = candidates.to_dict('records')
    
    # Save to new file
    output_file = log_dir / 'trades_candidates.pkl'
    with open(output_file, 'wb') as f:
        pickle.dump(candidates_list, f)
    
    print(f"✓ Exported {len(candidates_list)} candidate trades to {output_file.name}")
    print(f"\nNext step: Run the visualizer to label these trades:")
    print(f"  python trade_visualizer.py --trades {output_file}")

In [ ]:
## 9b. Interactive Trade Labeling Wizard

import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
from plotly.subplots import make_subplots

class TradeLabelingWizard:
    def __init__(self, trades_df, market_df):
        self.df = trades_df.copy()
        self.market_df = market_df  # Full OHLCV market data
        self.current_idx = 0
        
        # Initialize label and notes columns if they don't exist
        if 'label' not in self.df.columns:
            self.df['label'] = 'skip'
        if 'notes' not in self.df.columns:
            self.df['notes'] = ''
        
        # Convert any NaN to empty strings
        self.df['label'] = self.df['label'].fillna('skip')
        self.df['notes'] = self.df['notes'].fillna('')
        
        # Initialize VP calculator
        self.vp_calculator = VisibleRangeVP(n_bins=50)
        
        # Create widgets
        self.output = widgets.Output()
        self.chart_output = widgets.Output()
        self.label_dropdown = widgets.Dropdown(
            options=['good_entry', 'bad_entry', 'skip'],
            value='skip',
            description='Label:'
        )
        self.notes_text = widgets.Textarea(
            value='',
            placeholder='Add notes (optional)',
            description='Notes:',
            rows=2
        )
        self.prev_btn = widgets.Button(description='← Previous', button_style='info')
        self.next_btn = widgets.Button(description='Next →', button_style='success')
        self.save_btn = widgets.Button(description='💾 Save Labels', button_style='warning')
        self.delete_btn = widgets.Button(description='🗑️ Delete Trade', button_style='danger')
        self.progress_label = widgets.Label()
        
        # Button callbacks
        self.prev_btn.on_click(self.prev_trade)
        self.next_btn.on_click(self.next_trade)
        self.save_btn.on_click(self.save_labels)
        self.delete_btn.on_click(self.delete_trade)
        
        # Layout
        self.controls = widgets.HBox([self.prev_btn, self.next_btn, self.save_btn, self.delete_btn])
        self.ui = widgets.VBox([
            self.progress_label,
            self.output,
            self.chart_output,
            self.label_dropdown,
            self.notes_text,
            self.controls
        ])
    
    def show_trade(self):
        with self.output:
            clear_output(wait=True)
            
            if self.current_idx >= len(self.df):
                print("✅ All trades reviewed!")
                return
            
            trade = self.df.iloc[self.current_idx]
            
            # Update progress
            self.progress_label.value = f"Trade {self.current_idx + 1} / {len(self.df)}"
            
            # Display trade info
            print("="*60)
            print(f"TRADE #{self.current_idx} - {trade['action']}")
            print("="*60)
            print(f"📊 Entry: ${trade['entry_price']:.2f} @ step {trade['timestamp']}")
            print(f"📊 Exit:  ${trade['exit_price']:.2f}")
            print(f"💰 P&L:   ${trade['pnl']:.2f} ({trade['pnl_pct']:.2f}%)")
            print(f"⏱️  Duration: {trade['duration']} candles")
            
            # Load existing label if available
            label_val = str(trade['label']) if pd.notna(trade['label']) else 'skip'
            if label_val in ['good_entry', 'bad_entry', 'skip']:
                self.label_dropdown.value = label_val
            
            notes_val = str(trade['notes']) if pd.notna(trade['notes']) else ''
            self.notes_text.value = notes_val
        
        # Show chart
        self.show_chart(trade)
    
    def show_chart(self, trade):
        with self.chart_output:
            clear_output(wait=True)
            
            if self.market_df.empty:
                print("⚠️ No market data available for visualization")
                return
            
            # Get entry timestamp and calculate range
            timestamp = int(trade['timestamp'])
            lookback = 288  # VP lookback period
            
            # Extract OHLCV data for the visible range (288 candles before entry)
            start_idx = max(0, timestamp - lookback)
            end_idx = timestamp
            
            if start_idx >= len(self.market_df) or end_idx > len(self.market_df):
                print(f"⚠️ Trade timestamp {timestamp} out of range (market data: {len(self.market_df)} candles)")
                return
            
            # Get the visible range data
            visible_data = self.market_df.iloc[start_idx:end_idx].copy()
     
            volume_bins, vp_result = self.vp_calculator.calculate_vp(visible_data)
            
            # Extract VP metrics
            vah_price = vp_result['vah']
            poc_price = vp_result['poc']
            val_price = vp_result['val']
            vp_high = vp_result['high']
            vp_low = vp_result['low']
            
            # Get data for chart (show visible range + some lookahead)
            lookahead = min(int(trade['duration']) + 50, len(self.market_df) - end_idx)
            chart_start = max(0, start_idx - 50)  # Show some context before VP range
            chart_end = end_idx + lookahead
            chart_data = self.market_df.iloc[chart_start:chart_end].copy()
            
            # Adjust entry_idx to account for the extra context we're showing
            entry_offset = start_idx - chart_start
            
            # Create Plotly figure with subplots (price + RSI + volume profile)
            has_rsi = 'rsi' in chart_data.columns
            
            if has_rsi:
                fig = make_subplots(
                    rows=2, cols=2,
                    row_heights=[0.7, 0.3],
                    column_widths=[0.9, 0.1],
                    subplot_titles=('Price Chart', 'Volume Profile', 'RSI', ''),
                    horizontal_spacing=0.02,
                    vertical_spacing=0.08,
                    specs=[[{"secondary_y": False}, {"secondary_y": False}],
                           [{"secondary_y": False}, {"type": "xy"}]]
                )
            else:
                fig = make_subplots(
                    rows=1, cols=2,
                    column_widths=[0.9, 0.1],
                    subplot_titles=('Price Chart', 'Volume Profile'),
                    horizontal_spacing=0.02
                )
            
            # Add candlestick chart (use dates if available)
            use_dates = 'date' in chart_data.columns
            if use_dates:
                # Convert to datetime index or use pandas series directly for Plotly
                x_vals = pd.to_datetime(chart_data['date'])
            else:
                x_vals = list(range(len(chart_data)))
            
            fig.add_trace(
                go.Candlestick(
                    x=x_vals,
                    open=chart_data['open'],
                    high=chart_data['high'],
                    low=chart_data['low'],
                    close=chart_data['close'],
                    name='Price',
                    increasing_line_color='green',
                    decreasing_line_color='red'
                ),
                row=1, col=1
            )
            
            # Add EMA 9 and EMA 21 if available
            if 'ema_9' in chart_data.columns:
                fig.add_trace(
                    go.Scatter(
                        x=x_vals,
                        y=chart_data['ema_9'],
                        mode='lines',
                        name='EMA 9',
                        line=dict(color='cyan', width=1.5),
                        opacity=0.7
                    ),
                    row=1, col=1
                )
            
            if 'ema_21' in chart_data.columns:
                fig.add_trace(
                    go.Scatter(
                        x=x_vals,
                        y=chart_data['ema_21'],
                        mode='lines',
                        name='EMA 21',
                        line=dict(color='magenta', width=1.5),
                        opacity=0.7
                    ),
                    row=1, col=1
                )
            
            # Mark entry point (at the end of visible range)
            entry_idx = lookback + entry_offset
            if entry_idx < len(x_vals):
                # Use the datetime value at the entry index
                if use_dates:
                    entry_x = x_vals.iloc[entry_idx]
                else:
                    entry_x = entry_idx
                
                # Add entry marker as arrow
                entry_price = trade['entry_price']
                is_long = trade['action'] == 'LONG'
                
                # Arrow pointing up for LONG (green), down for SHORT (red)
                arrow_symbol = 'triangle-up' if is_long else 'triangle-down'
                arrow_color = 'green' if is_long else 'red'
                
                fig.add_trace(
                    go.Scatter(
                        x=[entry_x],
                        y=[entry_price],
                        mode='markers',
                        marker=dict(
                            symbol=arrow_symbol,
                            size=20,
                            color=arrow_color,
                            line=dict(color='white', width=2)
                        ),
                        name='Entry',
                        showlegend=False,
                        hovertemplate=f'Entry: ${entry_price:.2f}<extra></extra>'
                    ),
                    row=1, col=1
                )
            
            # Mark exit point
            exit_idx = entry_idx + int(trade['duration'])
            if exit_idx < len(x_vals):
                if use_dates:
                    exit_x = x_vals.iloc[exit_idx]
                else:
                    exit_x = exit_idx
                
                # Add exit marker as X
                exit_price = trade['exit_price']
                
                fig.add_trace(
                    go.Scatter(
                        x=[exit_x],
                        y=[exit_price],
                        mode='markers',
                        marker=dict(
                            symbol='x',
                            size=15,
                            color='purple',
                            line=dict(color='white', width=2)
                        ),
                        name='Exit',
                        showlegend=False,
                        hovertemplate=f'Exit: ${exit_price:.2f}<extra></extra>'
                    ),
                    row=1, col=1
                )
            
            # Draw VP levels on candlestick chart (no labels)
            fig.add_hline(
                y=vah_price,
                line_color="red",
                line_width=2,
                opacity=0.8,
                row=1, col=1
            )
            
            fig.add_hline(
                y=poc_price,
                line_color="yellow",
                line_width=3,
                opacity=0.9,
                row=1, col=1
            )
            
            fig.add_hline(
                y=val_price,
                line_color="green",
                line_width=2,
                opacity=0.8,
                row=1, col=1
            )
            
            # Shade Value Area
            fig.add_hrect(
                y0=val_price,
                y1=vah_price,
                fillcolor="gray",
                opacity=0.15,
                line_width=0,
                row=1, col=1
            )
            
            # Mark VP range (high/low of the 288 candles)
            fig.add_hline(
                y=vp_high,
                line_color="orange",
                line_width=1,
                line_dash="dot",
                opacity=0.5,
                row=1, col=1
            )
            
            fig.add_hline(
                y=vp_low,
                line_color="orange",
                line_width=1,
                line_dash="dot",
                opacity=0.5,
                row=1, col=1
            )
            
            # Add Volume Profile histogram (horizontal bars)
            bin_prices = np.linspace(vp_low, vp_high, len(volume_bins))
            fig.add_trace(
                go.Bar(
                    y=bin_prices,
                    x=volume_bins,
                    orientation='h',
                    marker=dict(color='lightblue', line=dict(color='blue', width=1)),
                    name='Volume',
                    showlegend=False
                ),
                row=1, col=2
            )
            
            # Add POC line on VP chart
            fig.add_hline(
                y=poc_price,
                line_color="yellow",
                line_width=3,
                opacity=0.9,
                row=1, col=2
            )
            
            # Shade VA on VP chart
            fig.add_hrect(
                y0=val_price,
                y1=vah_price,
                fillcolor="gray",
                opacity=0.2,
                line_width=0,
                row=1, col=2
            )
            
            # Add RSI subplot if available
            if has_rsi:
                fig.add_trace(
                    go.Scatter(
                        x=x_vals,
                        y=chart_data['rsi'],
                        mode='lines',
                        name='RSI',
                        line=dict(color='purple', width=2),
                        showlegend=False
                    ),
                    row=2, col=1
                )
                
                # Add RSI reference lines
                fig.add_hline(y=70, line_dash="dash", line_color="red", line_width=1, opacity=0.5, row=2, col=1)
                fig.add_hline(y=30, line_dash="dash", line_color="green", line_width=1, opacity=0.5, row=2, col=1)
                fig.add_hline(y=50, line_dash="dot", line_color="gray", line_width=1, opacity=0.3, row=2, col=1)
                
                # Update RSI y-axis
                fig.update_yaxes(title_text="RSI", range=[0, 100], side="left", row=2, col=1)
                fig.update_xaxes(title_text="Time", row=2, col=1)
            
            # Update layout
            height = 850 if has_rsi else 700
            fig.update_layout(
                title=f"{trade['action']} Trade | Duration: {trade['duration']} candles | VP Range: 288 candles",
                xaxis_rangeslider_visible=False,
                height=height,
                showlegend=False,
                hovermode='x unified'
            )
            
            # Update axes - price on left only
            fig.update_yaxes(title_text="Price ($)", side="left", row=1, col=1)
            fig.update_xaxes(title_text="Time", row=1, col=1)
            
            # Update VP subplot - hide y-axis labels
            fig.update_xaxes(title_text="Volume", row=1, col=2)
            fig.update_yaxes(title_text="", showticklabels=False, row=1, col=2)
            
            fig.show()
    
    def prev_trade(self, btn):
        self.save_current_label()
        if self.current_idx > 0:
            self.current_idx -= 1
            self.show_trade()
    
    def next_trade(self, btn):
        self.save_current_label()
        if self.current_idx < len(self.df) - 1:
            self.current_idx += 1
            self.show_trade()
    
    def delete_trade(self, btn):
        if self.current_idx < len(self.df):
            with self.output:
                print(f"\n🗑️ Marked trade #{self.current_idx} for deletion")
            # Mark as deleted
            idx = self.df.index[self.current_idx]
            self.df.at[idx, 'label'] = 'deleted'
            # Move to next
            if self.current_idx < len(self.df) - 1:
                self.current_idx += 1
                self.show_trade()
    
    def save_current_label(self):
        if self.current_idx < len(self.df):
            idx = self.df.index[self.current_idx]
            self.df.at[idx, 'label'] = self.label_dropdown.value
            self.df.at[idx, 'notes'] = self.notes_text.value
    
    def save_labels(self, btn):
        self.save_current_label()
        
        # Filter out deleted trades
        df_to_save = self.df[self.df['label'] != 'deleted'].copy()
        
        # Convert to list and save
        labeled_list = df_to_save.to_dict('records')
        output_file = log_dir / 'trades_labeled.pkl'
        
        with open(output_file, 'wb') as f:
            pickle.dump(labeled_list, f)
        
        labeled_count = len(df_to_save[df_to_save['label'].isin(['good_entry', 'bad_entry'])])
        deleted_count = len(self.df[self.df['label'] == 'deleted'])
        
        with self.output:
            clear_output(wait=True)
            print("="*60)
            print("✅ LABELS SAVED!")
            print("="*60)
            print(f"📁 File: {output_file.name}")
            print(f"📊 Labeled: {labeled_count} / {len(df_to_save)} trades")
            if deleted_count > 0:
                print(f"🗑️ Deleted: {deleted_count} trades")
            print()
            print("Next: Run cells 10-11 to analyze labeled trades")
    
    def start(self):
        display(self.ui)
        self.show_trade()

# Start the wizard
if not df_trades.empty and len(candidates) > 0:
    if df_market.empty:
        print("⚠️ Market data not loaded. Run cell 2b first!")
    else:
        print("🧙 Starting Trade Labeling Wizard...")
        print(f"📊 Using {len(df_market)} candles of market data")
        wizard = TradeLabelingWizard(candidates, df_market)
        wizard.start()
else:
    print("⚠️ No candidate trades available. Run cells 1-8 first.")

## 10. Analyze Labeled Trades (After Labeling)

Run this after you've labeled trades using the visualizer:

In [ ]:
# Load labeled trades
labeled_file = log_dir / 'trades_labeled.pkl'

if labeled_file.exists():
    with open(labeled_file, 'rb') as f:
        labeled_trades = pickle.load(f)
    
    df_labeled = pd.DataFrame(labeled_trades)
    
    print(f"✓ Loaded {len(df_labeled)} labeled trades")
    
    # Count labels
    label_counts = df_labeled['label'].value_counts()
    print(f"\nLabel distribution:")
    for label, count in label_counts.items():
        if label:
            print(f"  {label}: {count}")
    
    # Analyze good entries
    good_entries = df_labeled[df_labeled['label'] == 'good_entry']
    if len(good_entries) > 0:
        print(f"\n✅ Good Entries Analysis ({len(good_entries)} trades):")
        print(f"  Avg P&L: ${good_entries['pnl'].mean():.2f} ({good_entries['pnl_pct'].mean():.2f}%)")
        print(f"  Avg dist_to_poc: {good_entries['dist_to_poc'].mean():.3f}")
        print(f"  Avg dist_to_val: {good_entries['dist_to_val'].mean():.3f}")
        print(f"  Avg dist_to_vah: {good_entries['dist_to_vah'].mean():.3f}")
        print(f"  close_in_va rate: {good_entries['close_in_va'].mean()*100:.1f}%")
    
    # Analyze bad entries
    bad_entries = df_labeled[df_labeled['label'] == 'bad_entry']
    if len(bad_entries) > 0:
        print(f"\n❌ Bad Entries Analysis ({len(bad_entries)} trades):")
        print(f"  Avg P&L: ${bad_entries['pnl'].mean():.2f} ({bad_entries['pnl_pct'].mean():.2f}%)")
        print(f"  Avg dist_to_poc: {bad_entries['dist_to_poc'].mean():.3f}")
        print(f"  Avg dist_to_val: {bad_entries['dist_to_val'].mean():.3f}")
        print(f"  Avg dist_to_vah: {bad_entries['dist_to_vah'].mean():.3f}")
        print(f"  close_in_va rate: {bad_entries['close_in_va'].mean()*100:.1f}%")
    
    # Compare LONG vs SHORT patterns
    print(f"\n📊 By Action Type:")
    for action in ['LONG', 'SHORT']:
        action_good = good_entries[good_entries['action'] == action]
        if len(action_good) > 0:
            print(f"\n  Good {action} entries ({len(action_good)}):")
            print(f"    dist_to_poc: {action_good['dist_to_poc'].mean():.3f}")
            print(f"    dist_to_val: {action_good['dist_to_val'].mean():.3f}")
            print(f"    dist_to_vah: {action_good['dist_to_vah'].mean():.3f}")
            print(f"    close_above_poc: {action_good['close_above_poc'].mean()*100:.1f}%")
else:
    print("⚠️ No labeled trades file found.")
    print("Run: python trade_visualizer.py --trades logs/trades/trades_candidates.pkl")

## 11. Generate Reward Shaping Insights

In [ ]:
if labeled_file.exists() and len(good_entries) > 0 and len(bad_entries) > 0:
    print("="*60)
    print("REWARD SHAPING RECOMMENDATIONS")
    print("="*60)
    
    # LONG entry patterns
    good_longs = good_entries[good_entries['action'] == 'LONG']
    if len(good_longs) > 0:
        print("\n✅ REWARD Good LONG Entries When:")
        avg_poc = good_longs['dist_to_poc'].mean()
        avg_val = good_longs['dist_to_val'].mean()
        below_poc_rate = (1 - good_longs['close_above_poc'].mean()) * 100
        
        print(f"  - Distance to POC around: {avg_poc:.3f}")
        print(f"  - Distance to VAL around: {avg_val:.3f}")
        print(f"  - Below POC {below_poc_rate:.0f}% of the time")
        
        if avg_val < 0.1 and avg_val > -0.2:
            print(f"  → Pattern: Buying near VAL support ✓")
        if avg_poc < 0 or below_poc_rate > 60:
            print(f"  → Pattern: Buying below POC (support) ✓")
    
    # SHORT entry patterns
    good_shorts = good_entries[good_entries['action'] == 'SHORT']
    if len(good_shorts) > 0:
        print("\n✅ REWARD Good SHORT Entries When:")
        avg_poc = good_shorts['dist_to_poc'].mean()
        avg_vah = good_shorts['dist_to_vah'].mean()
        above_poc_rate = good_shorts['close_above_poc'].mean() * 100
        
        print(f"  - Distance to POC around: {avg_poc:.3f}")
        print(f"  - Distance to VAH around: {avg_vah:.3f}")
        print(f"  - Above POC {above_poc_rate:.0f}% of the time")
        
        if avg_vah < 0.1 and avg_vah > -0.1:
            print(f"  → Pattern: Selling near VAH resistance ✓")
        if avg_poc > 0 or above_poc_rate > 60:
            print(f"  → Pattern: Selling above POC (resistance) ✓")
    
    # Bad patterns
    bad_longs = bad_entries[bad_entries['action'] == 'LONG']
    if len(bad_longs) > 0:
        print("\n❌ PENALIZE Bad LONG Entries When:")
        avg_vah = bad_longs['dist_to_vah'].mean()
        above_poc_rate = bad_longs['close_above_poc'].mean() * 100
        
        if abs(avg_vah) < 0.15:
            print(f"  → Buying near VAH (resistance) - dist_to_vah: {avg_vah:.3f}")
        if above_poc_rate > 70:
            print(f"  → Buying way above POC - {above_poc_rate:.0f}% above POC")
    
    print("\n" + "="*60)
    print("Copy these patterns to trade_reward_shaper.py!")
    print("="*60)